In [ ]:
# %% Deep learning - Section 21.196
#    Code challenge 33: letters to numbers
#
#    1) Start from code from video 19.184 (emnist dataset)
#    2) Train a CNN to classify the letter MNIST dataset
#    3) Transfer the model to the digit MNIST dataset
#       - adapt the output layer (26 to 10)
#       - freeze all the convolutional layers, fine-tune the FC layers
#    4) Use 5 training epochs for the letter MNIST
#    5) Use 1 training epoch for the fine tuning
#    6) Try also fine-tuning the whole model (no freezing), also 1 epoch

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [2]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [10]:
# %% EMNIST data

# Load data
# See also : https://www.nist.gov/itl/products-and-services/emnist-dataset
data = torchvision.datasets.EMNIST(root='emnist',split='letters',download=True)


In [11]:
# %% Preprocess EMNIST

# Transform into 4D tensor for convolution layers, transform from int8 to float
images = data.data.view([124800,1,28,28]).float()

# Remove N/A (first class category; relabel to start at 0)
letter_categories = data.classes[1:]
labels = copy.deepcopy(data.targets)-1

# Normalise
images /= torch.max(images)


In [12]:
# %% Create train and test EMNIST datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size = 32
letters_train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
letters_test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [13]:
# %% MNIST data

# Load data
data = np.loadtxt(open('sample_data/mnist_train_small.csv','rb'),delimiter=',')

# Split labels from data
labels = data[:,0]
data   = data[:,1:]

# Normalise data (original range is (0,255))
data_norm = data / np.max(data)

# New here: reshape to 2D actual images
data_norm = data_norm.reshape(data_norm.shape[0],1,28,28)


In [14]:
# %% Create train and test MNIST datasets

# Convert to tensor (float and integers)
data_tensor   = torch.tensor(data_norm).float()
labels_tensor = torch.tensor(labels).long()

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(data_tensor,labels_tensor,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size = 32
numbers_train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
numbers_test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [34]:
# %% Function to generate the model

def gen_model(n_out_classes=26):

    class EMNIST_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Convolutional layers
            self.features = nn.Sequential( nn.Conv2d(1,6,kernel_size=3,stride=1,padding=1),
                                           nn.MaxPool2d(2),
                                           nn.BatchNorm2d(6),
                                           nn.LeakyReLU(),

                                           nn.Conv2d(6,6,kernel_size=3,stride=1,padding=1),
                                           nn.MaxPool2d(2),
                                           nn.BatchNorm2d(6),
                                           nn.LeakyReLU() )

            # Classifier (fully connected layers)
            self.classifier = nn.Sequential( nn.Linear(7*7*6, 50),
                                             nn.LeakyReLU(),
                                             nn.BatchNorm1d(50),
                                             nn.Linear(50, n_out_classes) )

        def forward(self,x):

            x = self.features(x)
            x = x.view(x.size(0),-1)
            x = self.classifier(x)

            return x

    # Create model instance
    CNN = EMNIST_CNN()

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [44]:
# %% Function to train the model

def train_model(CNN,train_loader,test_loader,optimizer,num_epochs=5):

    # Inizialise vars

    losses    = []
    train_acc = []
    test_acc  = []

    # Ship model to GPU
    CNN.to(device)

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append(loss.item())

            matches     = torch.argmax(yHat,axis=1) == y
            matches_num = matches.float()
            accuracy    = 100 * torch.mean(matches_num)
            batch_acc.append(accuracy)

        losses.append( np.mean(batch_loss) )
        train_acc.append( np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))

            # Ship to GPU
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)

            yHat = yHat.cpu()
            y    = y.cpu()

            test_acc.append( 100*torch.mean((torch.argmax(yHat,axis=1)==y).float()) )

        CNN.train()

    return train_acc,test_acc,losses,CNN


In [45]:
# %% Fit model on EMNIST

# Takes ~1 mins on GPU
letters_CNN,loss_fun,optimizer = gen_model(n_out_classes=26)
letters_train_acc,letters_test_acc,letters_losses,letters_CNN = train_model(letters_CNN,
                                                                            letters_train_loader,
                                                                            letters_test_loader,
                                                                            optimizer,
                                                                            num_epochs=5)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*5,5))

ax[0].plot(letters_losses,'s-')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')

ax[1].plot(letters_train_acc,'s-',label='Train')
ax[1].plot(letters_test_acc,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model test accuracy: {letters_test_acc[-1]:.2f}%')
ax[1].legend()

plt.savefig('figure4_code_challenge_33.png')
plt.show()
files.download('figure4_code_challenge_33.png')


In [ ]:
# %% Fine tune the FC layers (freeze convolutional layers)

# Fresh target model
numbers_CNN,loss_fun,_ = gen_model(n_out_classes=10)

# Load pretrained EMNIST weights from source model
state_dict = letters_CNN.state_dict()

# For the classifier, the 0 keys is the first FC layer, no 1 key (ReLU, no
# trainable params), key 2 is the batchnorm layer, key 3 is the final FC layer
for key in state_dict.keys():
    print(key)

# Remove last layer weights (since shape is different)
state_dict.pop('classifier.3.weight')
state_dict.pop('classifier.3.bias')

# Load remaining weights (strict=False to let PyTorch ignore the missing output
# layer; print out message about incompatible keys expected)
_ = numbers_CNN.load_state_dict(state_dict, strict=False)

# Freeze all the convolutional layers
for param in numbers_CNN.features.parameters():
    param.requires_grad = False

# Re-train only the classifier section of the model
optimizer = torch.optim.Adam(numbers_CNN.classifier.parameters(),lr=0.001)


In [ ]:
# %% Now fine-tune

numbers_train_acc,numbers_test_acc,numbers_losses,numbers_CNN = train_model(numbers_CNN,
                                                                            numbers_train_loader,
                                                                            numbers_test_loader,
                                                                            optimizer  = optimizer,
                                                                            num_epochs = 1)

print(f'MNIST train accuracy after EMNIST fine-tuning (FC tuning): {numbers_train_acc[-1]:.2f}%')
print(f'MNIST test accuracy after EMNIST fine-tuning (FC tuning): {numbers_test_acc[-1]:.2f}%')


In [ ]:
# %% Fine tune all the layers

# Fresh target model
numbers_CNN,loss_fun,_ = gen_model(n_out_classes=10)

# Load pretrained EMNIST weights from source model
state_dict = letters_CNN.state_dict()

# For the classifier, the 0 keys is the first FC layer, no 1 key (ReLU, no
# trainable params), key 2 is the batchnorm layer, key 3 is the final FC layer
for key in state_dict.keys():
    print(key)

# Remove last layer weights (since shape is different)
state_dict.pop('classifier.3.weight')
state_dict.pop('classifier.3.bias')

# Load remaining weights (strict=False to let PyTorch ignore the missing output
# layer; print out message about incompatible keys expected)
_ = numbers_CNN.load_state_dict(state_dict, strict=False)

# Un-freeze all the convolutional layers
for param in numbers_CNN.features.parameters():
    param.requires_grad = True

# Re-train only the classifier section of the model
optimizer = torch.optim.Adam(numbers_CNN.parameters(),lr=0.001)


In [ ]:
# %% Now fine-tune

numbers_train_acc,numbers_test_acc,numbers_losses,numbers_CNN = train_model(numbers_CNN,
                                                                            numbers_train_loader,
                                                                            numbers_test_loader,
                                                                            optimizer  = optimizer,
                                                                            num_epochs = 1)

print(f'MNIST train accuracy after EMNIST fine-tuning (full tuning): {numbers_train_acc[-1]:.2f}%')
print(f'MNIST test accuracy after EMNIST fine-tuning (full tuning): {numbers_test_acc[-1]:.2f}%')
